In [1]:
pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import PyPDF2
def load_and_clean_pdf(file_path):
   raw_text=""
   with open(file_path,'rb')as file:
    pdf_reader=PyPDF2.PdfReader(file)
    for page_num in range(len(pdf_reader.pages)):
        page=pdf_reader.pages[page_num]
        text=page.extract_text()
        if text:
            raw_text+=text+ " "
    cleaned_text=re.sub(r'(?i)^chapter\s+[a-z0-9]+$','',raw_text,flags=re.MULTILINE)
    cleaned_text=re.sub(r'[^a-zA-Z\s]','',cleaned_text)
    cleaned_text=cleaned_text.lower().strip()
    return cleaned_text

clean_text=load_and_clean_pdf('GOT.pdf')
print(f"total characters after extraction and cleaning:{len(clean_text)}")

unknown widths : 
[0, IndirectObject(1766, 0, 2338116112736)]


total characters after extraction and cleaning:1543587


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()
tokenizer.fit_on_texts([clean_text])
total_words=len(tokenizer.word_index)+1
print(f"Total unique words (Vocabulary Size):{total_words}")

Total unique words (Vocabulary Size):13354


In [4]:
token_list=tokenizer.texts_to_sequences([clean_text])[0]
sequence_length=10
input_sequences=[]
for i in range(sequence_length,len(token_list)):
    seq=token_list[i-sequence_length : i+1]
    input_sequences.append(seq)
print(f"Total sequences generated:{len(input_sequences)}")

Total sequences generated:295258


In [11]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

input_sequences=np.array(input_sequences)
X=input_sequences[:,:-1]
y=input_sequences[:,-1]


X.shape
y.shape

(295258,)

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [13]:
model=Sequential()
model.add(Embedding(total_words,100,input_length=sequence_length))
model.add(LSTM(150))
model.add(Dense(total_words,activation='softmax'))


c:\Users\SANSKAR\OneDrive\Desktop\myfolder\venv-tf\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X,y,epochs=100,batch_size=128)

Epoch 1/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 99s 42ms/step - accuracy: 0.0768 - loss: 6.4836
Epoch 2/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 95s 41ms/step - accuracy: 0.1154 - loss: 5.7790
Epoch 3/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 134s 38ms/step - accuracy: 0.1352 - loss: 5.4082
Epoch 4/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 87s 38ms/step - accuracy: 0.1484 - loss: 5.1455
Epoch 5/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 85s 37ms/step - accuracy: 0.1600 - loss: 4.9268
Epoch 6/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 142s 37ms/step - accuracy: 0.1707 - loss: 4.7327
Epoch 7/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 86s 37ms/step - accuracy: 0.1818 - loss: 4.5554
Epoch 8/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 91s 39ms/step - accuracy: 0.1934 - loss: 4.3888
Epoch 9/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 86s 37ms/step - accuracy: 0.2084 - loss: 4.2317
Epoch 10/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 85s 37ms/step - accuracy: 0.2232 - loss: 4.0819
Epoch 11/100
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 84s 36ms/step - accuracy: 0.2407 - loss: 3.

In [ ]:
import time
text =" what is the"
for i in range(10):
    token_text=tokenizer.texts_to_sequences([text])[0]
    padded_token_text=pad_sequences([token_text],maxlen=,padding='pre')
    pos=np.argmax(model.predict(padded_token_text))
    for word,index in tokenizer.word_index.items():
        if index==pos:
            text=text +" "+ word
            print(text)
            time.sleep(2)